In [2]:
import pandas as pd
import numpy as np
import sqlite3

# --- 1. VERİ ÜRETİMİ (Sentetik İş Verisi) ---
np.random.seed(42)
n_customers = 1000

# Müşteri ve Sözleşme Tablosu
customer_ids = [f"CUST-{1000+i}" for i in range(n_customers)]
contract_types = np.random.choice(['Aylık', '1 Yıllık', '2 Yıllık'], size=n_customers, p=[0.6, 0.3, 0.1])
payment_methods = np.random.choice(['Kredi Kartı', 'Banka Havalesi', 'Otomatik Ödeme'], size=n_customers)
monthly_charges = np.random.normal(loc=120, scale=35, size=n_customers).round(2)
monthly_charges = np.clip(monthly_charges, 30, 250)

df_customers = pd.DataFrame({
    'CustomerID': customer_ids,
    'ContractType': contract_types,
    'PaymentMethod': payment_methods,
    'MonthlyCharge_USD': monthly_charges
})

# Ürün Kullanım Tablosu
days_since_last_login = np.random.randint(0, 60, size=n_customers)
support_tickets = np.random.poisson(lam=2, size=n_customers)
usage_score = np.random.uniform(10, 100, size=n_customers).round(1)

df_usage = pd.DataFrame({
    'CustomerID': customer_ids,
    'DaysSinceLastLogin': days_since_last_login,
    'SupportTicketsCount': support_tickets,
    'UsageScore': usage_score
})

# Churn (İptal) Durumu Tablosu
churn_prob = (
    (df_customers['ContractType'] == 'Aylık').astype(int) * 0.35 +
    (df_usage['DaysSinceLastLogin'] > 30).astype(int) * 0.30 +
    (df_usage['SupportTicketsCount'] > 3).astype(int) * 0.20 +
    np.random.uniform(0, 0.15, size=n_customers)
)
churn_status = (churn_prob > 0.45).astype(int)

df_churn = pd.DataFrame({
    'CustomerID': customer_ids,
    'IsChurned': churn_status
})

# --- 2. GERÇEK SQL VERİTABANI DOSYASI OLUŞTURMA & YAZMA ---
# Bilgisayarında 'saas_database.db' adında kalıcı bir SQL dosyası oluşur
conn = sqlite3.connect("saas_database.db")

df_customers.to_sql("customers", conn, index=False, if_exists="replace")
df_usage.to_sql("usage", conn, index=False, if_exists="replace")
df_churn.to_sql("churn", conn, index=False, if_exists="replace")

# --- 3. MASTER TABLO OLUŞTURMA (SQL INNER JOIN) ---
query_join = """
SELECT 
    c.CustomerID,
    c.ContractType,
    c.PaymentMethod,
    c.MonthlyCharge_USD,
    u.DaysSinceLastLogin,
    u.SupportTicketsCount,
    u.UsageScore,
    ch.IsChurned
FROM customers c
INNER JOIN usage u ON c.CustomerID = u.CustomerID
INNER JOIN churn ch ON c.CustomerID = ch.CustomerID
"""

df_main = pd.read_sql_query(query_join, conn)

# --- 4. YÖNETİCİ CİRO HASAR RAPORU (SQL SORGUSU) ---
query_metrics = """
SELECT 
    COUNT(*) AS Toplam_Musteri,
    SUM(IsChurned) AS Kaybedilen_Musteri,
    ROUND(AVG(IsChurned) * 100, 2) || '%' AS Churn_Orani,
    '$' || ROUND(SUM(CASE WHEN IsChurned = 1 THEN MonthlyCharge_USD ELSE 0 END), 2) AS Kaybedilen_Aylik_MRR
FROM (
    SELECT c.MonthlyCharge_USD, ch.IsChurned
    FROM customers c
    JOIN churn ch ON c.CustomerID = ch.CustomerID
)
"""

df_metrics = pd.read_sql_query(query_metrics, conn)
print("--- YÖNETİCİ CİRO HASAR RAPORU (SQL SORGUSU SONUCU) ---")
display(df_metrics)

--- YÖNETİCİ CİRO HASAR RAPORU (SQL SORGUSU SONUCU) ---


,Toplam_Musteri,Kaybedilen_Musteri,Churn_Orani,Kaybedilen_Aylik_MRR
0,1000,421,42.1%,$51176.41


In [3]:
# --- 1. SÖZLEŞME TİPİNE GÖRE CHURN ORANI VE MRR KAYBI ---
query_contract = """
SELECT 
    c.ContractType AS Sozlesme_Tipi,
    COUNT(*) AS Toplam_Musteri,
    SUM(ch.IsChurned) AS Kaybedilen_Musteri,
    ROUND(AVG(ch.IsChurned) * 100, 2) || '%' AS Churn_Orani,
    '$' || ROUND(SUM(CASE WHEN ch.IsChurned = 1 THEN c.MonthlyCharge_USD ELSE 0 END), 2) AS Kaybedilen_MRR
FROM customers c
JOIN churn ch ON c.CustomerID = ch.CustomerID
GROUP BY c.ContractType
ORDER BY AVG(ch.IsChurned) DESC;
"""

# --- 2. KULLANIM VE DESTEK DAVRANIŞLARINA GÖRE DEĞERLENDİRME ---
query_behavior = """
SELECT 
    CASE WHEN ch.IsChurned = 1 THEN 'Terk Eden (Churn)' ELSE 'Kalan (Retained)' END AS Musteri_Durumu,
    ROUND(AVG(u.DaysSinceLastLogin), 1) AS Ort_Son_Giris_Gunu,
    ROUND(AVG(u.SupportTicketsCount), 1) AS Ort_Destek_Bileti,
    ROUND(AVG(u.UsageScore), 1) AS Ort_Kullanim_Skoru
FROM usage u
JOIN churn ch ON u.CustomerID = ch.CustomerID
GROUP BY ch.IsChurned;
"""

df_contract_analysis = pd.read_sql_query(query_contract, conn)
df_behavior_analysis = pd.read_sql_query(query_behavior, conn)

print("--- 1. SÖZLEŞME TİPİNE GÖRE KÖK NEDEN ANALİZİ ---")
display(df_contract_analysis)

print("\n--- 2. MÜŞTERİ DAVRANIŞLARINA GÖRE KÖK NEDEN ANALİZİ ---")
display(df_behavior_analysis)

--- 1. SÖZLEŞME TİPİNE GÖRE KÖK NEDEN ANALİZİ ---


,Sozlesme_Tipi,Toplam_Musteri,Kaybedilen_Musteri,Churn_Orani,Kaybedilen_MRR
0,Aylık,613,387,63.13%,$46933.76
1,1 Yıllık,287,26,9.06%,$3315.09
2,2 Yıllık,100,8,8.0%,$927.56



--- 2. MÜŞTERİ DAVRANIŞLARINA GÖRE KÖK NEDEN ANALİZİ ---


,Musteri_Durumu,Ort_Son_Giris_Gunu,Ort_Destek_Bileti,Ort_Kullanim_Skoru
0,Kalan (Retained),23.9,1.7,55.1
1,Terk Eden (Churn),37.5,2.3,52.6


In [4]:
# 1. Ödeme Yöntemine Göre Churn ve MRR Analizi
query_payment = """
SELECT 
    c.PaymentMethod AS Odeme_Yontemi,
    COUNT(*) AS Toplam_Musteri,
    SUM(ch.IsChurned) AS Kaybedilen_Musteri,
    ROUND(AVG(ch.IsChurned) * 100, 2) || '%' AS Churn_Orani,
    '$' || ROUND(SUM(CASE WHEN ch.IsChurned = 1 THEN c.MonthlyCharge_USD ELSE 0 END), 2) AS Kaybedilen_MRR
FROM customers c
JOIN churn ch ON c.CustomerID = ch.CustomerID
GROUP BY c.PaymentMethod
ORDER BY AVG(ch.IsChurned) DESC;
"""

# 2. Fiyat Segmentasyonuna (VIP vs Standart) Göre Churn ve MRR Analizi
query_price_segment = """
SELECT 
    CASE 
        WHEN c.MonthlyCharge_USD < 100 THEN '1. Düşük ( < $100 )'
        WHEN c.MonthlyCharge_USD BETWEEN 100 AND 150 THEN '2. Orta ( $100 - $150 )'
        ELSE '3. Yüksek/VIP ( > $150 )'
    END AS Fiyat_Segmenti,
    COUNT(*) AS Toplam_Musteri,
    SUM(ch.IsChurned) AS Kaybedilen_Musteri,
    ROUND(AVG(ch.IsChurned) * 100, 2) || '%' AS Churn_Orani,
    '$' || ROUND(SUM(CASE WHEN ch.IsChurned = 1 THEN c.MonthlyCharge_USD ELSE 0 END), 2) AS Kaybedilen_MRR,
    '$' || ROUND(AVG(c.MonthlyCharge_USD), 2) AS Ort_Aylik_Ucret
FROM customers c
JOIN churn ch ON c.CustomerID = ch.CustomerID
GROUP BY Fiyat_Segmenti
ORDER BY Fiyat_Segmenti;
"""

df_payment_analysis = pd.read_sql_query(query_payment, conn)
df_price_analysis = pd.read_sql_query(query_price_segment, conn)

print("--- 1. ÖDEME YÖNTEMİNE GÖRE ANALİZ ---")
display(df_payment_analysis)

print("\n--- 2. FIYAT SEGMENTİNE (VIP / STANDART) GÖRE ANALİZ ---")
display(df_price_analysis)

--- 1. ÖDEME YÖNTEMİNE GÖRE ANALİZ ---


,Odeme_Yontemi,Toplam_Musteri,Kaybedilen_Musteri,Churn_Orani,Kaybedilen_MRR
0,Kredi Kartı,341,148,43.4%,$18413.02
1,Banka Havalesi,343,148,43.15%,$17105.24
2,Otomatik Ödeme,316,125,39.56%,$15658.15



--- 2. FIYAT SEGMENTİNE (VIP / STANDART) GÖRE ANALİZ ---


,Fiyat_Segmenti,Toplam_Musteri,Kaybedilen_Musteri,Churn_Orani,Kaybedilen_MRR,Ort_Aylik_Ucret
0,1. Düşük ( < $100 ),268,114,42.54%,$9176.0,$78.96
1,2. Orta ( $100 - $150 ),532,224,42.11%,$28002.69,$124.12
2,3. Yüksek/VIP ( > $150 ),200,83,41.5%,$13997.72,$169.01
